In [8]:
import numpy as np
import pandas as pd
from scipy import stats
import xarray as xr

import s3fs
import pyarrow as pa
import pyarrow.parquet as pq
from utils import remove_outliers, WMO

fs = s3fs.S3FileSystem(anon=True)

In [9]:
def slide(x, k, fun, n=1, **kwargs):
    """
    Apply a function in a sliding window along a vector.
    
    Allows to compute a moving average, moving median, or even moving standard 
    deviation, etc. in a generic way.
    
    Parameters:
    -----------
    x : array-like
        Input numeric vector
    k : int
        Order of the window; the window size is 2k+1
    fun : callable
        Function to apply in the moving window
    n : int, optional
        Number of times to pass the function over the data (default=1)
    **kwargs : dict
        Arguments passed to fun (e.g., na.rm equivalent would be handled by nanmean, nanmedian, etc.)
    
    Returns:
    --------
    np.ndarray
        The data passed through fun, n times
    """
    x = np.array(x, dtype=float)
    
    if n >= 1:
        for t in range(n):
            # Pad the extremities of data with NaN
            x_padded = np.concatenate([np.full(k, np.nan), x, np.full(k, np.nan)])
            
            # Apply the rolling function
            result = []
            for i in range(k, len(x_padded) - k):
                window = x_padded[(i-k):(i+k+1)]
                result.append(fun(window, **kwargs))
            
            x = np.array(result)
    
    return x


def despike(x, k=3, method='median', threshold=2):
    """
    Despike data using a reference calculated with a moving window.
    
    This is a simplified version inspired by oce::despike. The oce package uses
    a more sophisticated approach, but this captures the essential functionality.
    
    Parameters:
    -----------
    x : array-like
        Input numeric vector
    k : int
        Order of the window; the window size is 2k+1 (default=3)
    method : str
        Method for reference calculation: 'median' or 'mean' (default='median')
    threshold : float
        Number of standard deviations for spike detection (default=2)
    
    Returns:
    --------
    np.ndarray
        Despiked data
    """
    x = np.array(x, dtype=float)
    
    # Calculate reference using sliding window
    if method == 'median':
        reference = slide(x, k, np.nanmedian)
    else:
        reference = slide(x, k, np.nanmean)
    
    # Calculate residuals
    residuals = x - reference
    
    # Calculate MAD (Median Absolute Deviation) for robust threshold
    mad = np.nanmedian(np.abs(residuals - np.nanmedian(residuals)))
    
    # Identify spikes (using MAD-based threshold, more robust than SD)
    threshold_value = threshold * mad * 1.4826  # 1.4826 converts MAD to SD equivalent
    is_spike = np.abs(residuals) > threshold_value
    
    # Replace spikes with reference values
    x_despiked = x.copy()
    x_despiked[is_spike] = reference[is_spike]
    
    return x_despiked

In [33]:
def extract_cp_data(wmo, ds):
    """
    Extract transmissometer data from netCDF file.
    
    Parameters:
    -----------
    wmo : str
        WMO float identifier
    ds
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with CP660 data
    """
    
    # Extract data
    value = ds['CP660'].values
    depth = ds['PRES'].values
    mc = ds['MEASUREMENT_CODE'].values
    juld = ds['JULD'].values
    cycle = ds['CYCLE_NUMBER'].values
    
    # Create DataFrame
    df = pd.DataFrame({
        'wmo': wmo,
        'cycle': cycle,
        'juld': juld,
        'mc': mc,
        'depth': depth,
        'cp': value,
    })
    
    # Clean data
    df = df[
        (df['cycle'] >= 1) & 
        (df['mc'] == 290)
    ].dropna(subset=['cp']).copy()
    
    # Compute park_depth
    df['park_depth'] = df['depth'].apply(
        lambda x: 200 if x < 350 else (1000 if x > 750 else 500)
    )
    
    df = df.drop(columns=['mc'])
    
    return df

In [34]:
def derive_ost_flux(data, wmo_float):
    """
    Derive optical sediment trap flux from CP data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame containing CP data
    wmo_float : str
        WMO float identifier
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with computed fluxes
    """
    # Filter and sort data chronologically
    tmp = data[data['wmo'] == wmo_float].copy()
    tmp = tmp.sort_values('juld').reset_index(drop=True)
    
    # Despike cp data with a 7-point moving window (k=3)
    tmp['cp'] = despike(tmp['cp'].values, k=3)
    
    # Smooth cp data with a 3-point moving median, n time(s)
    tmp['cp'] = slide(tmp['cp'].values, fun=np.nanmedian, k=3, n=1)
    
    # Compute slope between two adjacent points
    delta_x = tmp['juld'].diff().dt.total_seconds() / (24 * 3600)  # Convert to days
    delta_y = tmp['cp'].diff()
    tmp['slope'] = delta_y / delta_x
    
    # Compute Z score on the slopes
    mean_slope = tmp['slope'].mean()
    std_slope = tmp['slope'].std()
    tmp['zscore'] = (tmp['slope'] - mean_slope) / std_slope
    
    # Spot outliers using IQR method
    Q1 = tmp['zscore'].quantile(0.25)
    Q3 = tmp['zscore'].quantile(0.75)
    IQR = Q3 - Q1
    
    spikes_down = tmp['zscore'] < (Q1 - 1.5 * IQR)
    spikes_up = tmp['zscore'] > (Q3 + 1.5 * IQR)
    spikes = spikes_down | spikes_up
    
    tmp['spikes'] = spikes
    
    # Assign colour code to cp signal
    tmp['colour'] = 'base signal'
    tmp.loc[tmp['spikes'], 'colour'] = 'jump'
    
    # Add group to compute the slope of each group of points
    tmp['group'] = None
    
    # Index of jumps
    jump_index = tmp[tmp['colour'] == 'jump'].index.tolist()
    
    # Assign group identity
    for i in jump_index:
        mask = (tmp.index < i) & (tmp['group'].isna())
        tmp.loc[mask, 'group'] = f'group_{i}'
    
    tmp.loc[tmp['group'].isna(), 'group'] = 'last_group'
    
    # Compute slope for each subgroup
    slope_list = []
    for group_name, group_data in tmp[tmp['colour'] == 'base signal'].groupby('group'):
        group_data = group_data.dropna(subset=['slope'])
        if len(group_data) > 3:
            min_time = group_data['juld'].min()
            max_time = group_data['juld'].max()
            nb_points = len(group_data)
            first_cp = group_data.iloc[0]['cp']
            last_cp = group_data.iloc[-1]['cp']
            delta_x = (max_time - min_time).total_seconds() / (24 * 3600)
            delta_y = (last_cp - first_cp) * 0.25  # Convert cp to ATN
            slope = delta_y / delta_x
            
            if slope > 0:  # Remove negative slopes
                slope_list.append({
                    'group': group_name,
                    'min_time': min_time,
                    'max_time': max_time,
                    'nb_points': nb_points,
                    'first_cp': first_cp,
                    'last_cp': last_cp,
                    'delta_x': delta_x,
                    'delta_y': delta_y,
                    'slope': slope
                })
    
    slope_df = pd.DataFrame(slope_list)
    
    if len(slope_df) > 0:
        # Compute weighted average slope
        mean_slope = (slope_df['nb_points'] * slope_df['slope']).sum() / slope_df['nb_points'].sum()
        
        # Convert cp to POC using Estapa's relationship
        poc_flux = 633 * (mean_slope ** 0.77)
    else:
        poc_flux = 0
    
    # Build dataframe to plot each subgroup
    part1 = slope_df[['group', 'min_time', 'first_cp']].rename(
        columns={'min_time': 'time', 'first_cp': 'cp'}
    )
    part2 = slope_df[['group', 'max_time', 'last_cp']].rename(
        columns={'max_time': 'time', 'last_cp': 'cp'}
    )
    part_slope = pd.concat([part1, part2], ignore_index=True)
    
    # Spot negative jump
    negative_jump_mask = (tmp['colour'] == 'jump') & (tmp['slope'] < 0)
    tmp.loc[negative_jump_mask, 'colour'] = 'negative jump'
    
    # Add large particles flux
    rows_to_keep = []
    for idx in jump_index:
        if idx > 0:
            rows_to_keep.extend([idx-1, idx])
    
    tmp2 = tmp.loc[rows_to_keep, ['juld', 'cp', 'slope', 'colour', 'group']].sort_values('juld')
    
    # Remove negative jumps, if any
    check_colour = tmp2['colour'].unique()
    if len(check_colour) >= 2:  # At least one jump
        tmp2['diff_jump'] = tmp2['cp'].diff()
        tmp2 = tmp2.iloc[1::2]  # Keep even indexes
    else:
        tmp2 = None
    
    if tmp2 is None or len(tmp2) == 0:
        large_part_poc_flux = 0
        tmp3 = None
    else:
        tmp3 = tmp2[tmp2['diff_jump'] > 0]
        if len(tmp3) == 0:
            large_part_poc_flux = 0
        else:
            delta_y = tmp3['diff_jump'].sum() * 0.25  # Convert to ATN
            max_time = tmp['juld'].max()
            min_time = tmp['juld'].min()
            delta_x = (max_time - min_time).total_seconds() / (24 * 3600)
            slope_large_part = delta_y / delta_x
            large_part_poc_flux = 633 * (slope_large_part ** 0.77)
    
    # Compute total drifting time
    max_time = tmp['juld'].max()
    min_time = tmp['juld'].min()
    drifting_time = (max_time - min_time).total_seconds() / (24 * 3600)
    
    # Create result DataFrame
    result = pd.DataFrame({
        'max_time': [max_time],
        'min_time': [min_time],
        'small_flux': [poc_flux],
        'large_flux': [large_part_poc_flux],
        'park_depth': [data['park_depth'].iloc[0]],
        'wmo': [data['wmo'].iloc[0]],
        'cycle': [data['cycle'].iloc[0]]
    })
    
    return result

In [16]:
# Extract particle data for each float
dfs = []

WMO = [1902578]

for wmo in WMO:
    try:
        with fs.open(f"s3://argo-gdac-sandbox/pub/dac/coriolis/{wmo}/{wmo}_Rtraj.nc", 'rb') as f:
            ds = xr.open_dataset(f)
            # dfs.append(df)
    except Exception as e:
        print(f"Error processing {wmo}: {e}", file=sys.stderr)
        continue

/tmp/ipykernel_12421/3010766494.py:9: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(f)


nan

In [46]:
df = extract_cp_data(wmo, ds)

In [47]:
df

,wmo,cycle,juld,depth,cp,park_depth
1168,1902578,1.0,2022-05-29 19:03:19.000000000,971.200012,0.018401,1000
1173,1902578,1.0,2022-05-29 19:33:18.000000000,971.200012,0.018088,1000
1176,1902578,1.0,2022-05-29 20:03:18.000000000,978.000000,0.019339,1000
1180,1902578,1.0,2022-05-29 20:33:17.999999744,978.000000,0.019339,1000
1183,1902578,1.0,2022-05-29 21:03:18.000000000,981.700012,0.019339,1000
...,...,...,...,...,...,...
169679,1902578,39.0,2023-04-08 23:55:37.000000000,969.099976,0.310459,1000
169682,1902578,39.0,2023-04-09 00:25:37.000000000,969.099976,0.313150,1000
169684,1902578,39.0,2023-04-09 00:55:37.000000000,969.099976,0.357136,1000
169687,1902578,39.0,2023-04-09 01:25:37.000000000,969.099976,0.356116,1000


In [48]:
derive_ost_flux(df, wmo)

,max_time,min_time,small_flux,large_flux,park_depth,wmo,cycle
0,2023-04-09 01:55:37,2022-05-29 19:03:19,4.925671,15.667071,1000,1902578,1.0


In [37]:
df

,wmo,cycle,juld,depth,cp,park_depth
1168,1902578,1.0,2022-05-29 19:03:19.000000000,971.200012,0.018401,1000
1173,1902578,1.0,2022-05-29 19:33:18.000000000,971.200012,0.018088,1000
1176,1902578,1.0,2022-05-29 20:03:18.000000000,978.000000,0.019339,1000
1180,1902578,1.0,2022-05-29 20:33:17.999999744,978.000000,0.019339,1000
1183,1902578,1.0,2022-05-29 21:03:18.000000000,981.700012,0.019339,1000
...,...,...,...,...,...,...
169679,1902578,39.0,2023-04-08 23:55:37.000000000,969.099976,0.310459,1000
169682,1902578,39.0,2023-04-09 00:25:37.000000000,969.099976,0.313150,1000
169684,1902578,39.0,2023-04-09 00:55:37.000000000,969.099976,0.357136,1000
169687,1902578,39.0,2023-04-09 01:25:37.000000000,969.099976,0.356116,1000


In [60]:
def extract_ost_data(wmo, ds):
    """
    Extract optical sediment trap data.
    
    Parameters:
    -----------
    wmo_float : int or str
        WMO float identifier
    path_to_data : str
        Path to data directory
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with OST flux data
    """
    # Parking depths
    park_depths = [200, 500, 1000]
    
    # Extract cp data from the float
    data = extract_cp_data(wmo, ds)

    
    res = []
    max_cycle = data['cycle'].max()
    
    for park_depth in park_depths:
        for cycle in range(1, int(max_cycle) + 1):
            #print(park_depth, cycle)
            tmp = data[(data['park_depth'] == park_depth) & (data['cycle'] == cycle)]
            #print(tmp)
            
            if len(tmp) == 0:  # No data for this cycle or at this parking depth
                continue
            elif len(tmp) < 3:  # Not enough data
                continue
            else:
                try: 
                    output = derive_ost_flux(tmp, wmo)
                    res.append(output)
                except:
                    continue
                    
    
    if len(res) > 0:
        return pd.concat(res, ignore_index=True)
    else:
        return pd.DataFrame()

In [61]:
tmp

,max_time,min_time,small_flux,large_flux,park_depth,wmo,cycle
0,2022-06-01 05:57:23,2022-05-31 16:27:23.000000000,15.124741,6.338278,200,1902578,3.0
1,2022-06-04 07:32:22,2022-06-03 18:02:22.000000000,11.808719,13.888895,200,1902578,5.0
2,2022-06-06 17:12:37,2022-06-05 17:42:38.000000000,6.944044,15.574690,200,1902578,6.0
3,2022-06-16 17:59:43,2022-06-15 18:29:43.000000000,17.281964,11.349071,200,1902578,7.0
4,2022-06-25 17:37:03,2022-06-24 18:07:02.999999744,7.769147,15.197811,200,1902578,8.0
...,...,...,...,...,...,...,...
89,2023-03-01 10:54:57,2023-02-20 00:55:09.000000000,3.944480,6.065003,1000,1902578,35.0
90,2023-03-11 11:12:36,2023-03-02 01:12:49.000000000,3.780019,6.446129,1000,1902578,36.0
91,2023-03-21 11:11:39,2023-03-12 00:11:52.000000000,4.081963,2.948658,1000,1902578,37.0
92,2023-03-31 10:48:04,2023-03-25 12:18:12.000000000,3.468690,1.248008,1000,1902578,38.0


In [59]:
# Process all floats
results = []
for wmo in WMO:
    result = extract_ost_data(wmo, ds)
    if len(result) > 0:
        results.append(result)

tmp = pd.concat(results, ignore_index=True)

200 1
200 2
200 3
200 4
200 5
200 6
200 7
200 8
200 9
200 10
200 11
200 12
200 13
200 14
200 15
200 16
200 17
200 18
200 19
200 20
200 21
200 22
200 23
200 24
200 25
200 26
200 27
200 28
200 29
200 30
200 31
200 32
200 33
200 34
200 35
200 36
200 37
200 38
200 39
500 1
500 2
500 3
500 4
500 5
500 6
500 7
500 8
500 9
500 10
500 11
500 12
500 13
500 14
500 15
500 16
500 17
500 18
500 19
500 20
500 21
500 22
500 23
500 24
500 25
500 26
500 27
500 28
500 29
500 30
500 31
500 32
500 33
500 34
500 35
500 36
500 37
500 38
500 39
1000 1
1000 2
1000 3
1000 4
1000 5
1000 6
1000 7
1000 8
1000 9
1000 10
1000 11
1000 12
1000 13
1000 14
1000 15
1000 16
1000 17
1000 18
1000 19
1000 20
1000 21
1000 22
1000 23
1000 24
1000 25
1000 26
1000 27
1000 28
1000 29
1000 30
1000 31
1000 32
1000 33
1000 34
1000 35
1000 36
1000 37
1000 38
1000 39


In [55]:
tmp

NameError: name 'tmp' is not defined